# InsideForest: caso de uso reproducible

Este notebook es la versión canónica incluida en la propia librería. Presenta:

1. clustering supervisado por regiones con Iris;
2. utilidades que suelen quedar fuera de una demostración corta, como importancias y persistencia;
3. clustering regional guiado por clase con Wine (tres clases);
4. regiones representativas, ambigüedad, distribución de clases y filas sin asignar;
5. un ejemplo breve de regresión.

Todos los datasets vienen con scikit-learn. No se necesita una API key ni descargar archivos de datos.


In [ ]:
# En Colab instala InsideForest solamente cuando todavía no está disponible.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("InsideForest") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "InsideForest>=0.4.0"]
    )


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from InsideForest import (
    InsideForestClassRegionClusterer,
    InsideForestContinuousRegionClusterer,
    InsideForestRegionClusterer,
)
from InsideForest.multiclass import (
    extract_multiclass_leaf_rules,
    get_multiclass_labels,
)

try:
    insideforest_version = version("InsideForest")
except PackageNotFoundError:
    insideforest_version = "código fuente local"

print("InsideForest:", insideforest_version)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 20)


# 1. Clustering supervisado por regiones con Iris

`InsideForestRegionClusterer` usa el objetivo para descubrir regiones y devuelve **etiquetas de cluster**.
Estas etiquetas no son IDs de clase. `score` mide AMI, `assign_regions` explica la asignación y una fila sin coincidencias recibe `-1`.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X_iris = iris.data.copy()
X_iris.columns = [
    column.replace(" (cm)", "").replace(" ", "_")
    for column in X_iris.columns
]
y_iris = iris.target.to_numpy()

iris_frame = X_iris.copy()
iris_frame["species"] = y_iris
iris_frame.head()


In [ ]:
traditional = InsideForestRegionClusterer(
    rf_params={
        "n_estimators": 16,
        "max_depth": 5,
        "random_state": 15,
        "n_jobs": 1,
    },
    tree_params={
        "lang": "py",
        "n_sample_multiplier": 0.05,
        "ef_sample_multiplier": 10,
    },
    var_obj="species",
    no_trees_search=16,
    leaf_percentile=95,
    low_leaf_fraction=0.10,
    max_cases=len(X_iris),
    get_detail=True,
    seed=15,
)

iris_cluster_labels = traditional.fit_predict(X_iris, y_iris)
iris_quality = traditional.region_quality_report(X_iris, y_iris)

pd.Series(
    {
        "ami": traditional.score(X_iris, y_iris),
        "coverage": iris_quality["coverage"],
        "unmatched_rate": iris_quality["unmatched_rate"],
        "n_regions": len(traditional.regions_),
    },
    name="Iris",
)


In [ ]:
cluster_table = (
    pd.DataFrame({"species": y_iris, "cluster": iris_cluster_labels})
    .groupby(["species", "cluster"])
    .size()
    .unstack(fill_value=0)
)

ax = cluster_table.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
    colormap="viridis",
)
ax.set(
    title="Iris: clase real frente a cluster supervisado",
    xlabel="Clase real",
    ylabel="Observaciones",
)
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
traditional.explain_regions(top_n=10)[[
    "cluster_id", "region_target_class", "support", "coverage",
    "dominant_probability", "lift", "entropy", "region_score",
]]


In [ ]:
iris_assignments = traditional.assign_regions(X_iris)
iris_assignments[[
    "cluster_id", "region_target_class", "membership_score",
    "target_probability", "lift", "entropy",
    "matched_region_count", "source",
]].head(10)


## Importancias y persistencia

El clusterer expone las importancias del bosque **generador de ramas** y conserva regiones, métricas y asignaciones al guardarse.


In [ ]:
importance_table = (
    pd.DataFrame(
        {
            "feature": traditional.feature_names_in_,
            "importance": traditional.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

ax = importance_table.plot.bar(
    x="feature",
    y="importance",
    legend=False,
    figsize=(8, 4),
    color="#4C78A8",
)
ax.set(title="Importancias del Random Forest", ylabel="Importancia")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

importance_table


In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    model_path = Path(tmp_dir) / "insideforest_iris.joblib"
    traditional.save(model_path)
    restored = InsideForestRegionClusterer.load(model_path)
    restored_labels = restored.predict(X_iris)

print(
    "Asignaciones idénticas después de guardar/cargar:",
    np.array_equal(iris_cluster_labels, restored_labels),
)


# 2. Clustering supervisado por clase: Wine

Aquí usamos `InsideForestClassRegionClusterer`. El bosque sólo genera ramas candidatas:

- cada hoja conserva la distribución completa de clases;
- cada región se asocia con la clase que maximiza el objetivo supervisado;
- la salida es un ID de cluster regional, no una clase predicha;
- las filas sin región reciben `-1`, sin fallback.

Wine tiene tres clases y trece variables numéricas.


In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine = wine.data.copy()
y_wine = np.asarray(
    [wine.target_names[index] for index in wine.target],
    dtype=object,
)

X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine,
    y_wine,
    test_size=0.25,
    stratify=y_wine,
    random_state=42,
)

pd.Series(y_wine_train).value_counts().sort_index()


In [ ]:
multiclass_model = InsideForestClassRegionClusterer(
    rf_params={
        "n_estimators": 24,
        "max_depth": 4,
        "min_samples_leaf": 4,
        "random_state": 42,
        "n_jobs": 1,
    },
    leaf_percentile=90,
    low_leaf_fraction=0.20,
    min_support=3,
    max_regions_per_class=60,
    branch_aggregation="none",
    random_state=42,
    n_jobs=1,
    ambiguity_margin=0.35,
)

multiclass_model.fit(X_wine_train, y_wine_train)

pd.Series(
    {
        "classes": list(multiclass_model.classes_),
        "n_regions": len(multiclass_model.regions_),
        "classes_in_regions": multiclass_model.regions_["region_target_class"].nunique(),
    },
    name="Wine",
)


## Regiones por clase

Cada región contiene pureza, cobertura, lift y `class_distribution`. `region_target_class` es metadato supervisado y no una predicción.


In [ ]:
def format_distribution(probabilities, classes):
    return {str(label): round(float(probability), 3) for label, probability in zip(classes, probabilities)}

top_regions = multiclass_model.explain_regions(top_n=12).copy()
top_regions["class_distribution"] = top_regions["class_distribution"].map(
    lambda values: format_distribution(values, multiclass_model.classes_)
)

top_regions[[
    "cluster_id", "region_target_class", "support", "coverage",
    "target_probability", "lift", "entropy", "region_score",
    "class_distribution", "description",
]]


In [ ]:
region_metrics = multiclass_model.region_metrics_.copy()
region_metrics["class_distribution"] = region_metrics["class_distribution"].map(
    lambda values: format_distribution(values, multiclass_model.classes_)
)

region_metrics.sort_values(["target_probability", "support"], ascending=False).head(10)


## Asignación sobre datos no vistos

`assign_regions` devuelve el cluster regional elegido o `-1`; nunca delega la fila al bosque.


In [ ]:
wine_membership = multiclass_model.transform(X_wine_test)
wine_assignments = multiclass_model.assign_regions(X_wine_test)
wine_results = wine_assignments.assign(actual_class=y_wine_test)

metrics = pd.Series(multiclass_model.region_quality_report(X_wine_test, y_wine_test), name="holdout")
covered = wine_assignments["cluster_id"].to_numpy() != -1
argmax_matches_predict = np.array_equal(
    wine_membership[covered].argmax(axis=1),
    wine_assignments.loc[covered, "cluster_id"].to_numpy(),
)

display(metrics[["coverage", "unmatched_rate", "cluster_purity", "nmi", "ami", "ari"]])
display(multiclass_model.class_coverage_report())
print("transform.argmax reproduce predict en filas cubiertas:", argmax_matches_predict)
wine_results.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

wine_results["source"].value_counts().plot.bar(ax=axes[0], color="#2E86AB")
axes[0].set(title="Cobertura de regiones", xlabel="Fuente", ylabel="Filas")

covered = wine_results[wine_results["source"] == "region"]
covered["class_margin"].plot.hist(ax=axes[1], bins=12, color="#5B8E7D")
axes[1].axvline(multiclass_model.ambiguity_margin, color="crimson", linestyle="--")
axes[1].set(title="Margen de clase en regiones cubiertas", xlabel="Margen")

plt.tight_layout()
plt.show()


## Regiones representativas y ambiguas

Las primeras maximizan el objetivo para una clase; las ambiguas tienen poco margen entre las dos clases principales.


In [ ]:
representative = pd.concat(
    [multiclass_model.regions_for_class(label, top_n=3) for label in multiclass_model.classes_],
    ignore_index=True,
)
representative_view = representative[[
    "region_target_class", "support", "target_probability", "lift", "region_score", "description"
]].sort_values(["region_target_class", "region_score"], ascending=[True, False])

display(representative_view)

ax = representative_view.plot.barh(y="region_score", legend=False, figsize=(9, 5), color="#5B8E7D")
ax.invert_yaxis()
ax.set(title="Score de regiones representativas", xlabel="Score", ylabel="Regi?n")
plt.tight_layout()
plt.show()


In [ ]:
ambiguous_regions = multiclass_model.ambiguous_regions(top_n=10)

if ambiguous_regions.empty:
    print("No se encontraron regiones ambiguas con el umbral actual.")
else:
    display(ambiguous_regions[[
        "cluster_id", "region_target_class", "target_probability",
        "class_margin", "support", "description",
    ]])


## Filas sin región

Al desactivar temporalmente las regiones, todas las filas quedan en el cluster `-1`; no existe fallback al bosque.


In [ ]:
selected_regions = multiclass_model.regions_
try:
    multiclass_model.regions_ = selected_regions.iloc[0:0].copy()
    unmatched_demo = multiclass_model.assign_regions(X_wine_test.head(2))
finally:
    multiclass_model.regions_ = selected_regions

unmatched_demo


## API de bajo nivel

La fachada `InsideForest.multiclass` también permite extraer reglas y recalcular sus etiquetas
cuando se necesita construir un flujo personalizado.


In [ ]:
custom_rules = extract_multiclass_leaf_rules(
    multiclass_model.forest_,
    X_wine_train,
    y_wine_train,
    percentil=95,
    low_frac=0.05,
    min_support=3,
    max_rules_per_class=10,
    random_state=42,
)

custom_labels = get_multiclass_labels(
    custom_rules,
    X_wine_train,
    y_wine_train,
    class_labels=multiclass_model.classes_,
)

custom_labels[
    [
        "target_class",
        "support",
        "coverage",
        "target_probability",
        "lift",
        "entropy",
        "description",
    ]
].sort_values(["target_class", "target_probability"], ascending=[True, False]).head(12)


# 3. Clustering regional para un objetivo continuo

`InsideForestContinuousRegionClusterer` usa el bosque sólo para generar hojas. `predict` devuelve IDs regionales, `score` mide η² e incluye el cluster `-1`.


In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    diabetes.data,
    diabetes.target,
    test_size=0.25,
    random_state=42,
)

continuous_model = InsideForestContinuousRegionClusterer(
    auto_feature_reduce=True,
    explicit_k_features=8,
    rf_params={
        "n_estimators": 50,
        "max_depth": 6,
        "min_samples_leaf": 3,
        "random_state": 42,
        "n_jobs": 1,
    },
    leaf_percentile=90,
    low_leaf_fraction=0.12,
    min_support=3,
    branch_aggregation="none",
    max_cases=len(X_reg_train),
    random_state=42,
    n_jobs=1,
)
continuous_model.fit(X_reg_train, y_reg_train)

continuous_regions = continuous_model.predict(X_reg_test)
continuous_assignments = continuous_model.assign_regions(X_reg_test)
continuous_membership = continuous_model.transform(X_reg_test)
region_report = continuous_model.region_quality_report(X_reg_test, y_reg_test)
pd.Series(
    {
        "eta_squared": continuous_model.score(X_reg_test, y_reg_test),
        "assigned_regions": pd.Series(continuous_regions).nunique(),
        "coverage": region_report["coverage"],
        "unmatched_rate": region_report["unmatched_rate"],
        "target_std_reduction": region_report["target_std_reduction"],
        "n_regions": region_report["n_regions"],
    },
    name="Diabetes",
)


## Validación de regiones continuas

El script `experiments/validate_regression_regions.py` evalúa `InsideForestContinuousRegionClusterer` como clustering supervisado continuo sobre Diabetes, Friedman1 y señales sintéticas.

El criterio principal es η² junto con cobertura, tasa sin asignar, reducción de dispersión, compresión y estabilidad. R²/RMSE del bosque se reportan por separado como diagnóstico del generador.


In [ ]:
from pathlib import Path

summary_path = Path("experiments/results/regression_region_validation/summary.csv")
if not summary_path.exists():
    summary_path = Path("../../experiments/results/regression_region_validation/summary.csv")

validation_summary = pd.read_csv(summary_path)
validation_summary[
    [
        "dataset",
        "runs",
        "test_eta_squared_median",
        "region_rmse_lift_vs_mean_median",
        "test_rule_coverage_median",
        "test_known_region_coverage_median",
        "test_target_std_reduction_median",
        "n_regions_median",
    ]
].round(4)


# Siguientes pasos

- Usa `InsideForestRegionClusterer` para el flujo regional tradicional.
- Usa `InsideForestClassRegionClusterer` para regiones enriquecidas y asociadas con clases nominales.
- Usa `InsideForestContinuousRegionClusterer` cuando el objetivo sea numérico.
- En los tres casos, `predict` devuelve clusters y nunca la predicción del bosque.
- Para validación, revisa `README.multiclass.md` y `experiments/validate_class_region_clusters.py`.
- Para regresión, revisa `experiments/validate_regression_regions.py`.
